In [0]:
from pyspark.sql.functions import *

In [0]:
history = spark.read.format("delta").load(
    "dbfs:/EmployeeLifecycleProject/history"
)

display(history)

emp_id,name,department,salary,join_date,status,effective_start_date,effective_end_date,is_current
1,Emp_1,SALES,46632,2020-07-30,ACTIVE,day_1,day_2,false
2,Emp_2,HR,65513,2021-06-19,ACTIVE,day_1,day_2,false
3,Emp_3,SALES,84005,2020-09-01,ACTIVE,day_1,day_2,false
4,Emp_4,HR,34931,2021-09-02,ACTIVE,day_1,day_2,false
5,Emp_5,HR,95521,2021-09-18,ACTIVE,day_1,day_3,false
6,Emp_6,IT,69478,2020-12-22,ACTIVE,day_1,day_4,false
7,Emp_7,SALES,30599,2021-07-25,ACTIVE,day_1,day_3,false
8,Emp_8,HR,55153,2020-04-30,ACTIVE,day_1,day_2,false
9,Emp_9,MARKETING,42541,2021-09-25,ACTIVE,day_1,day_2,false
10,Emp_10,FINANCE,96524,2020-08-12,ACTIVE,day_1,day_2,false


In [0]:
history.createOrReplaceTempView("employee_history")

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW current_employees AS
SELECT *
FROM employee_history
WHERE is_current = true;

In [0]:
%sql

SELECT * FROM current_employees;

emp_id,name,department,salary,join_date,status,effective_start_date,effective_end_date,is_current
19,Emp_19,IT,36875,2022-08-05,ACTIVE,day_1,null,true
29,Emp_29,HR,39637,2021-04-27,ACTIVE,day_1,null,true
31,Emp_31,SALES,48997,2020-08-09,ACTIVE,day_1,null,true
33,Emp_33,SALES,78953,2020-05-19,ACTIVE,day_1,null,true
36,Emp_36,FINANCE,46673,2020-02-02,ACTIVE,day_1,null,true
55,Emp_55,HR,40890,2021-07-25,ACTIVE,day_1,null,true
62,Emp_62,SALES,90786,2022-03-01,ACTIVE,day_1,null,true
87,Emp_87,FINANCE,38601,2020-07-11,ACTIVE,day_1,null,true
108,Emp_108,IT,49443,2020-12-27,ACTIVE,day_1,null,true
132,Emp_132,IT,72777,2020-02-26,ACTIVE,day_1,null,true


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW department_summary AS
SELECT department, COUNT(*) AS total_employees, ROUND(AVG(salary),2) AS average_salary, MIN(salary) AS minimum_salary, MAX(salary) AS maximum_salary
FROM current_employees
GROUP BY department
ORDER BY total_employees DESC;

In [0]:
%sql

SELECT * FROM department_summary;

department,total_employees,average_salary,minimum_salary,maximum_salary
HR,50,68943.48,30599,109476
SALES,49,74341.37,35357,104106
IT,48,69291.0,36278,100557
FINANCE,41,67050.17,31690,109271
MARKETING,32,73406.19,39904,106012


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW employee_status AS
SELECT status, COUNT(*) AS employee_count
FROM current_employees
GROUP BY status;

In [0]:
%sql

SELECT * FROM employee_status;

status,employee_count
ACTIVE,196
RESIGNED,24


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW salary_summary AS
SELECT MIN(salary) AS minimum_salary, MAX(salary) AS maximum_salary, ROUND(AVG(salary),2) AS average_salary
FROM current_employees;

In [0]:
%sql

SELECT * FROM salary_summary;

minimum_salary,maximum_salary,average_salary
30599,109476,70517.84


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW salary_band_summary AS
SELECT emp_id, name, department, salary,

CASE

WHEN salary < 40000
THEN 'Low Salary'

WHEN salary BETWEEN 40000 AND 70000
THEN 'Medium Salary'

WHEN salary BETWEEN 70001 AND 90000
THEN 'High Salary'

ELSE 'Very High Salary'
END AS salary_band
FROM current_employees;

In [0]:
%sql

SELECT * FROM salary_band_summary;

emp_id,name,department,salary,salary_band
19,Emp_19,IT,36875,Low Salary
29,Emp_29,HR,39637,Low Salary
31,Emp_31,SALES,48997,Medium Salary
33,Emp_33,SALES,78953,High Salary
36,Emp_36,FINANCE,46673,Medium Salary
55,Emp_55,HR,40890,Medium Salary
62,Emp_62,SALES,90786,Very High Salary
87,Emp_87,FINANCE,38601,Low Salary
108,Emp_108,IT,49443,Medium Salary
132,Emp_132,IT,72777,High Salary


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dense_salary_rank AS

SELECT emp_id, department, salary,

DENSE_RANK() OVER
(
ORDER BY salary DESC
) AS dense_rank

FROM current_employees;

In [0]:
%sql

SELECT * FROM dense_salary_rank;

emp_id,department,salary,dense_rank
5,HR,109476,1
99,HR,109412,2
109,FINANCE,109271,3
13,HR,108863,4
119,HR,107681,5
152,MARKETING,106012,6
179,MARKETING,104958,7
48,HR,104430,8
70,SALES,104106,9
92,HR,103892,10


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW top3_department_salary AS

SELECT *
FROM
(

SELECT emp_id, name, department, salary,

ROW_NUMBER() OVER
(
PARTITION BY department
ORDER BY salary DESC
) rn

FROM current_employees

)

WHERE rn<=3;

In [0]:
%sql 

SELECT * FROM top3_department_salary;

emp_id,name,department,salary,rn
109,Emp_109,FINANCE,109271,1
93,Emp_93,FINANCE,102379,2
169,Emp_169,FINANCE,101256,3
5,Emp_5,HR,109476,1
99,Emp_99,HR,109412,2
13,Emp_13,HR,108863,3
67,Emp_67,IT,100557,1
124,Emp_124,IT,100259,2
102,Emp_102,IT,99489,3
152,Emp_152,MARKETING,106012,1


In [0]:
current_emp_df= spark.sql("SELECT * FROM current_employees")

department_df = spark.sql("SELECT * FROM department_summary")

salary_sum_df= spark.sql("SELECT * FROM salary_summary")

salary_band_df = spark.sql("SELECT * FROM salary_band_summary")

salary_rank_df = spark.sql("SELECT * FROM dense_salary_rank")

top3_df = spark.sql("SELECT * FROM top3_department_salary")

status_df = spark.sql("SELECT * FROM employee_status")

In [0]:
current_emp_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/current_employees")

In [0]:
department_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/department_summary")

In [0]:
salary_sum_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/salary_summary")

In [0]:
salary_band_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/salary_band_summary")

In [0]:
salary_rank_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/dense_salary_rank")

In [0]:
top3_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/top3_department_salary")

In [0]:
status_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/employee_status")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/employeelifecycleworkspace/default/employee_outputs/gold"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/current_employees/,current_employees/,0,1784440261000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/dense_salary_rank/,dense_salary_rank/,0,1784440356000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/department_summary/,department_summary/,0,1784440273000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/employee_status/,employee_status/,0,1784440387000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/salary_band_summary/,salary_band_summary/,0,1784440344000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/salary_summary/,salary_summary/,0,1784440293000
dbfs:/Volumes/employeelifecycleworkspace/default/employee_outputs/gold/top3_department_salary/,top3_department_salary/,0,1784440371000
